# AdCanvas 메인 노트북 (배포용)

사진 한 장 → **① 누끼** → ② VLM(요청 해석) → ③ 배경 생성 → ④ 원본 음식 재합성

* 각 단계는 앞 단계 결과를 정해진 형식(`handoff`)으로 받아서 씀
* 평가(점수, 비교 실험)는 서브 노트북에서 함. 이 노트북은 **실행만**
* 런타임: GPU(T4) 권장. 허깅페이스 토큰 필요 없음

# 0. 설치

In [ ]:
!SAM2_BUILD_CUDA=0 pip install -q git+https://github.com/facebookresearch/sam2.git
!pip install -q pillow-heif qwen-vl-utils peft

# 1. 누끼 (Segmentation)

**사진 → 그릇 후보 → (사용자 선택) → 음식 마스크 + 투명 배경 누끼**

| 단계 | 모델 | 역할 |
|---|---|---|
| 그릇 찾기 | Grounding DINO tiny | 글자(plate, bowl …)로 그릇·음식 박스 찾기 |
| 그릇별 마스크 | SAM 2.1 hiera-small + **우리 파인튜닝 디코더** | 박스 하나 → 그릇 + 음식 + 걸친 수저를 통째로 |
| 후처리 | 직접 작성 | 조각 정리, 구멍 채우기, 중복 그릇 제거 |

* 파인튜닝 가중치: 드라이브 `AdCanvas_nukki/weights/sam2_ft.pt` (없으면 원래 SAM으로 동작)
* 결과 마스크: **흰색(255) = 음식(그릇 포함)**, 원본 사진과 같은 크기

## 1-1. 설정
* `NUKKI_WEIGHTS`: 파인튜닝 가중치 위치 (팀 드라이브 공유 폴더 기준)
* `OUT_DIR`: 결과 저장 위치 (다음 단계가 여기서 읽음)

In [ ]:
import os, io, json, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

from google.colab import drive
drive.mount('/content/drive')

NUKKI_BASE = '/content/drive/MyDrive/AdCanvas_nukki'
NUKKI_WEIGHTS = f'{NUKKI_BASE}/weights/sam2_ft.pt'
OUT_DIR = '/content/adcanvas_out'
os.makedirs(OUT_DIR, exist_ok=True)
print('파인튜닝 가중치:', '있음' if os.path.exists(NUKKI_WEIGHTS) else '없음 (원래 SAM 사용)')

## 1-2. 누끼 함수
실행만 하면 됨. 바깥에서 쓰는 함수:

| 함수 | 입력 → 출력 |
|---|---|
| `load()` | 모델 불러오기 (한 번) |
| `candidates(img)` | 사진 → 그릇 후보 목록 `[{'id','label','score','kind','dish','box','mask'}]` |
| `merge(img, cands, ids)` | 고른 후보만 합친 마스크 (ids=None이면 기본 선택 전부) |
| `segment(img)` | 사진 → 마스크 (자동, 기본 선택 전부) |
| `refine(img, box, points, labels)` | 사용자 박스·클릭 보정 → 그릇 하나 마스크 |
| `cutout(img, mask)` | 투명 배경 누끼 (RGBA) |

In [ ]:
import os

import cv2
import numpy as np
import torch
from PIL import Image, ImageOps
from scipy import ndimage

SAM_MODEL = 'facebook/sam2.1-hiera-small'
GD_ID = 'IDEA-Research/grounding-dino-tiny'
MAX_SIDE = 2048            # 이보다 큰 사진은 줄여서 처리하고, 마스크는 원래 크기로 되돌림

# 1_pipeline.ipynb와 같은 값 (11/16 test IoU 0.906, BF 0.827)
DINO_CFG = dict(
    dish_words=['plate', 'bowl', 'dish', 'pot'],            # 그릇 (기본 선택)
    other_words=['cup', 'glass', 'spoon', 'chopsticks'],    # 후보로만 (기본 선택 해제)
    food_words=['food'],                                    # 그릇 없이 놓인 음식
    box_th=0.3, nms_iou=0.6, max_area=0.9, food_max_area=0.5,
)
CAND = dict(
    utensil_words=['spoon', 'chopsticks'],
    utensil_overlap=0.2,   # 수저의 20% 이상이 그릇 외곽 안에 있으면 걸친 것 (원래 SAM일 때만 합침)
    part_min=0.1,          # 마스크 조각 중 가장 큰 조각의 10% 이상만 남김
    solidity=0.9,          # 원래 SAM일 때: 그릇이 볼록 외곽의 90% 미만이면 외곽으로 채움
    inside_drop=0.9,       # 다른 그릇 안에 90% 이상 들어간 후보는 중복으로 제거
    sam_chunk=4,           # SAM을 박스 몇 개씩 돌릴지 (메모리 절약)
)

_m = {}   # 불러온 모델: device, gd_proc, gd, sam, finetuned


# ---------------- 불러오기 ----------------
def _device():
    if torch.cuda.is_available():
        return 'cuda'
    if torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'


def load(weights=NUKKI_WEIGHTS):
    """모델 불러오기 (서버 켤 때 한 번). 반환: 파인튜닝 가중치를 썼는지"""
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    try:
        from pillow_heif import register_heif_opener
        register_heif_opener()   # 아이폰 HEIC
    except ImportError:
        pass
    dev = _device()
    _m['device'] = dev
    _m['gd_proc'] = AutoProcessor.from_pretrained(GD_ID)
    _m['gd'] = AutoModelForZeroShotObjectDetection.from_pretrained(GD_ID).to(dev).eval()
    sam = SAM2ImagePredictor.from_pretrained(SAM_MODEL, device=dev)
    _m['finetuned'] = bool(weights) and os.path.exists(weights)
    if _m['finetuned']:
        res = sam.model.load_state_dict(torch.load(weights, map_location=dev), strict=False)
        assert not res.missing_keys and not res.unexpected_keys, 'sam2_ft.pt와 SAM_MODEL이 맞지 않음'
    _m['sam'] = sam
    _m['sam_img'] = None   # set_image 한 사진 (클릭 보정 때 다시 계산하지 않으려고)
    return _m['finetuned']


def _ready():
    if 'sam' not in _m:
        load()


def _prep(img):
    """PIL 사진 → (처리용 RGB 사진, 원래 크기). EXIF 회전 반영, 긴 변 MAX_SIDE로"""
    img = ImageOps.exif_transpose(img).convert('RGB')
    size = img.size
    if max(size) > MAX_SIDE:
        img = img.copy()
        img.thumbnail((MAX_SIDE, MAX_SIDE), Image.LANCZOS)
    return img, size


def _to_size(m, size):
    """bool 마스크 → 원래 크기의 흑백 PIL (L, 0/255)"""
    out = Image.fromarray((m * 255).astype(np.uint8))
    return out if out.size == size else out.resize(size, Image.NEAREST)


def _set_image(img_np):
    key = (img_np.shape, int(img_np[::97, ::89].sum()))
    if _m['sam_img'] != key:
        _m['sam'].set_image(img_np)
        _m['sam_img'] = key


# ---------------- DINO ----------------
def _box_area(b):
    return (b[2] - b[0]) * (b[3] - b[1])


@torch.no_grad()
def dino_boxes(img):
    """사진 → [{'box', 'label', 'score', 'kind'(dish/other/food), 'dish'(기본 선택)}]"""
    from torchvision.ops import nms
    C, proc, gd = DINO_CFG, _m['gd_proc'], _m['gd']
    words = C['dish_words'] + C['other_words'] + C['food_words']
    kinds = ['dish'] * len(C['dish_words']) + ['other'] * len(C['other_words']) + ['food'] * len(C['food_words'])
    inputs = proc(images=img, text='. '.join(words) + '.', return_tensors='pt').to(_m['device'])
    out = gd(**inputs)
    probs = out.logits[0].sigmoid().float().cpu()
    ids = inputs['input_ids'][0].tolist()
    cols = []
    for w in words:
        sub = proc.tokenizer(w, add_special_tokens=False)['input_ids']
        pos = [i + j for i in range(len(ids) - len(sub) + 1) if ids[i:i + len(sub)] == sub for j in range(len(sub))]
        cols.append(probs[:, pos].max(1).values if pos else torch.zeros(len(probs)))
    score, best = torch.stack(cols, 1).max(1)
    W, H = img.size
    cx, cy, bw, bh = out.pred_boxes[0].float().cpu().unbind(-1)
    boxes = torch.stack([(cx - bw / 2) * W, (cy - bh / 2) * H, (cx + bw / 2) * W, (cy + bh / 2) * H], -1)
    boxes[:, 0::2] = boxes[:, 0::2].clamp(0, W)
    boxes[:, 1::2] = boxes[:, 1::2].clamp(0, H)
    keep = [i for i in range(len(boxes)) if score[i] > C['box_th'] and _box_area(boxes[i].tolist()) <
            (C['food_max_area'] if kinds[int(best[i])] == 'food' else C['max_area']) * W * H]
    if not keep:
        return []
    keep = torch.tensor(keep)
    k = keep[nms(boxes[keep], score[keep], C['nms_iou'])].tolist()
    return [{'box': boxes[i].tolist(), 'label': words[int(best[i])], 'score': float(score[i]),
             'kind': kinds[int(best[i])], 'dish': kinds[int(best[i])] != 'other'} for i in k]


# ---------------- 마스크 도우미 ----------------
def _box_mask(shape, box, expand=0.1):
    x0, y0, x1, y1 = box
    ex, ey = expand * (x1 - x0), expand * (y1 - y0)
    out = np.zeros(shape, bool)
    out[max(int(y0 - ey), 0):max(int(np.ceil(y1 + ey)), 0), max(int(x0 - ex), 0):max(int(np.ceil(x1 + ex)), 0)] = True
    return out


def _local(fn, m, pad=0):
    out = np.zeros_like(m)
    ys, xs = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
    if not len(ys):
        return out
    y0, y1 = max(ys[0] - pad, 0), min(ys[-1] + 1 + pad, m.shape[0])
    x0, x1 = max(xs[0] - pad, 0), min(xs[-1] + 1 + pad, m.shape[1])
    out[y0:y1, x0:x1] = fn(m[y0:y1, x0:x1])
    return out


def _hull(m):
    def f(c):
        out = np.zeros(c.shape, np.uint8)
        cv2.fillConvexPoly(out, cv2.convexHull(cv2.findNonZero(c.astype(np.uint8))), 1)
        return out.astype(bool)
    return _local(f, m)


def _close_fill(m):
    return _local(lambda c: ndimage.binary_fill_holes(ndimage.binary_closing(c, iterations=3)), m, pad=4)


def _keep_parts(c, box):
    lab, n = ndimage.label(c)
    if n <= 1:
        return c
    idx = np.arange(1, n + 1)
    sizes = ndimage.sum(c, lab, idx)
    inside = ndimage.sum(_box_mask(c.shape, box), lab, idx)
    keep = idx[(sizes >= CAND['part_min'] * sizes.max()) & (inside >= 0.5 * sizes)]
    return np.isin(lab, keep if len(keep) else idx[[sizes.argmax()]])


def _crop(m, box, fn, pad=4):
    ys, xs = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
    if not len(ys):
        return m
    y0, y1 = max(ys[0] - pad, 0), min(ys[-1] + 1 + pad, m.shape[0])
    x0, x1 = max(xs[0] - pad, 0), min(xs[-1] + 1 + pad, m.shape[1])
    out = np.zeros_like(m)
    out[y0:y1, x0:x1] = fn(m[y0:y1, x0:x1], [box[0] - x0, box[1] - y0, box[2] - x0, box[3] - y0])
    return out


def _pick_whole(ms, ss, box, mm_score=0.6, mm_inside=0.9):
    ok = [j for j in range(len(ms)) if ms[j].any() and ss[j] >= mm_score * ss.max()]
    bm = _box_mask(ms[0].shape, box)
    ok = [j for j in ok if (ms[j] & bm).sum() >= mm_inside * ms[j].sum()]
    return ms[max(ok, key=lambda j: ms[j].sum())] if ok else ms[int(ss.argmax())]


def _clean_dish(m, box):
    out = _crop(m, box, lambda c, b: ndimage.binary_fill_holes(ndimage.binary_closing(_keep_parts(c, b), iterations=3)))
    if not out.any():
        return out
    h = _hull(out) & (_box_mask(m.shape, box, 0.05) | out)
    return h if out.sum() < CAND['solidity'] * h.sum() else out


# ---------------- SAM + 후처리 ----------------
@torch.no_grad()
def _sam_masks(img_np, boxes, kinds):
    """파인튜닝 SAM은 마스크 1개, 원래 SAM은 3개 중 전체 물체를 고름"""
    _set_image(img_np)
    H, W = img_np.shape[:2]
    multi = not _m['finetuned']
    out = []
    for s0 in range(0, len(boxes), CAND['sam_chunk']):
        b = np.array(boxes[s0:s0 + CAND['sam_chunk']], np.float32)
        m, sc, _ = _m['sam'].predict(box=b, multimask_output=multi)
        m = np.asarray(m).reshape(len(b), -1, H, W) > 0.5
        if not multi:
            out += list(m[:, 0])
            continue
        sc = np.asarray(sc).reshape(len(b), -1)
        for k in range(len(b)):
            out.append(m[k, int(sc[k].argmax())] if kinds[s0 + k] == 'other' else _pick_whole(m[k], sc[k], boxes[s0 + k]))
    return out


def _tidy(cands, light):
    for c in cands:
        if c['dish'] and c['kind'] != 'food':
            c['mask'] = _close_fill(_crop(c['mask'], c['box'], _keep_parts, pad=0)) if light else _clean_dish(c['mask'], c['box'])
    dishes = sorted([c for c in cands if c['dish'] and c['kind'] != 'food'], key=lambda c: -c['mask'].sum())
    for i, small in enumerate(dishes):
        for big in dishes[:i]:
            if 'dup_of' not in big and (small['mask'] & big['mask']).sum() >= CAND['inside_drop'] * max(small['mask'].sum(), 1):
                small['dish'], small['dup_of'] = False, big['id']
                break
    kept = [d for d in dishes if 'dup_of' not in d]
    union = np.zeros(cands[0]['mask'].shape, bool)
    for d in kept:
        union |= d['mask']
    for f in [c for c in cands if c['kind'] == 'food']:
        f['mask'] = _close_fill(_crop(f['mask'], f['box'], _keep_parts, pad=0))
        a = f['mask'].sum()
        swallows = any((f['mask'] & d['mask']).sum() > 0.3 * d['mask'].sum() for d in kept)
        if a == 0 or (f['mask'] & union).sum() >= 0.5 * a or swallows:
            f['dish'], f['dup_of'] = False, -1
    utensils = [u for u in cands if u['label'] in CAND['utensil_words'] and u['mask'].any()]
    if utensils and not light:
        hulls = {c['id']: _hull(c['mask']) for c in cands if c['dish'] and 'dup_of' not in c and c['mask'].any()}
        for u in utensils:
            if not hulls:
                break
            best = max(hulls, key=lambda i: (u['mask'] & hulls[i]).sum())
            if (u['mask'] & hulls[best]).sum() > CAND['utensil_overlap'] * u['mask'].sum():
                d = next(c for c in cands if c['id'] == best)
                d['mask'] = _close_fill(d['mask'] | u['mask'])
                u['merged_into'] = best
    return [c for c in cands if 'dup_of' not in c]


# ---------------- 바깥에서 쓰는 함수 ----------------
def candidates(img):
    """사진 → 그릇 후보 목록. 박스는 원래 사진 좌표, mask는 원래 크기의 흑백 PIL
    [{'id', 'label', 'score', 'kind'(dish/other/food), 'dish'(기본 선택), 'box'[x0,y0,x1,y1], 'mask'}]
    그릇을 하나도 못 찾으면 [] (서비스에서는 사용자가 직접 박스를 그리게 → refine)"""
    _ready()
    work, size = _prep(img)
    found = dino_boxes(work)
    if not any(f['dish'] for f in found):
        return []
    masks = _sam_masks(np.array(work), [f['box'] for f in found], [f['kind'] for f in found])
    for i, (f, m) in enumerate(zip(found, masks)):
        f.update(id=i, mask=m)
    cands = _tidy(found, light=_m['finetuned'])
    s = size[0] / work.size[0]
    return [{'id': c['id'], 'label': c['label'], 'score': round(c['score'], 3), 'kind': c['kind'], 'dish': c['dish'],
             'box': [round(v * s, 1) for v in c['box']], 'mask': _to_size(c['mask'], size)} for c in cands]


def merge(img, cands, ids=None):
    """후보 중 고른 것만 합친 마스크 (ids=None이면 기본 선택된 그릇 전부)"""
    W, H = ImageOps.exif_transpose(img).size
    out = np.zeros((H, W), bool)
    for c in cands:
        if (c['id'] in ids) if ids is not None else c['dish']:
            out |= np.asarray(c['mask']) > 127
    return _to_size(out, (W, H))


def segment(img):
    """팀 공통 함수 모양 (KAN-5): RGB 사진 → 흑백 마스크 (L, 사진과 같은 크기, 255 = 음식+그릇)"""
    return merge(img, candidates(img))


@torch.no_grad()
def refine(img, box, points=None, labels=None):
    """클릭 보정: 박스(원래 사진 좌표) + 클릭 점(1 포함, 0 제외) → 그 그릇 하나의 마스크 (L, 원래 크기)"""
    _ready()
    work, size = _prep(img)
    s = work.size[0] / size[0]
    _set_image(np.array(work))
    kw = dict(box=np.array([v * s for v in box], np.float32), multimask_output=False)
    if points:
        kw.update(point_coords=np.array(points, np.float32) * s, point_labels=np.array(labels, np.int32))
    m, _, _ = _m['sam'].predict(**kw)
    return _to_size(np.asarray(m)[0] > 0.5, size)


def cutout(img, mask, crop=False):
    """원본 + 마스크 → 투명 배경 누끼 (RGBA). crop=True면 음식 부분만 잘라냄"""
    rgb = ImageOps.exif_transpose(img).convert('RGB')
    out = rgb.copy()
    out.putalpha(mask.convert('L').resize(rgb.size, Image.NEAREST))
    return out.crop(out.getbbox()) if crop and out.getbbox() else out

## 1-3. 모델 불러오기

In [ ]:
t0 = time.time()
ft = load(NUKKI_WEIGHTS)
print(f"device: {_m['device']} | 파인튜닝 SAM: {ft} | {time.time() - t0:.1f}초")

## 1-4. 사진 넣기
* `IMG_PATH`에 경로를 넣거나, 비워 두면 업로드 창이 뜸

In [ ]:
IMG_PATH = ''   # 예: f'{NUKKI_BASE}/dataset/images/hwang/IMG_0405.HEIC'

if not IMG_PATH:
    from google.colab import files
    up = files.upload()
    IMG_PATH = f'/content/{next(iter(up))}'
img = ImageOps.exif_transpose(Image.open(IMG_PATH)).convert('RGB')
print(IMG_PATH, img.size)

## 1-5. 그릇 후보 찾기
* 초록 = 그릇(기본 선택), 주황 = 컵·수저 등(기본 선택 안 함)
* 그릇을 하나도 못 찾으면 1-7(직접 박스)로

In [ ]:
COLORS = np.array([(230, 25, 75), (60, 180, 75), (0, 130, 200), (245, 130, 48), (145, 30, 180),
                   (70, 240, 240), (240, 50, 230), (210, 245, 60), (250, 190, 212), (0, 128, 128)])


def show_candidates(img, cands):
    base = np.array(img)
    over = base.copy()
    fig, ax = plt.subplots(1, 2, figsize=(14, 7))
    ax[0].imshow(base)
    for k, c in enumerate(cands):
        x0, y0, x1, y1 = c['box']
        col = 'lime' if c['dish'] else 'orange'
        ax[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color=col, lw=2))
        ax[0].text(x0, y0, f"{c['id']}:{c['label']} {c['score']:.2f}", fontsize=8, backgroundcolor=col)
        if c['dish']:
            m = np.asarray(c['mask']) > 127
            over[m] = (over[m] * 0.45 + COLORS[k % len(COLORS)] * 0.55).astype(np.uint8)
    ax[0].set_title('candidates (green = dish)'); ax[1].imshow(over); ax[1].set_title('dish candidates')
    for a in ax:
        a.axis('off')
    plt.tight_layout(); plt.show()


t0 = time.time()
cands = candidates(img)
print(f"후보 {len(cands)}개, 그릇 {sum(c['dish'] for c in cands)}개 | {time.time() - t0:.1f}초")
print('그릇 id:', [c['id'] for c in cands if c['dish']])
if cands:
    show_candidates(img, cands)
else:
    print('그릇을 찾지 못했어요. 1-7에서 박스를 직접 지정하세요.')

## 1-6. 그릇 고르기 → 마스크, 누끼
* `SELECT = None`: 기본 선택(그릇 전부) / `SELECT = [0, 2]`: 고른 id만

In [ ]:
SELECT = None

mask = merge(img, cands, SELECT) if cands else None

## 1-7. (선택) 직접 박스·클릭으로 보정
* 자동으로 못 찾았거나 고치고 싶을 때만. `BOX = [x0, y0, x1, y1]` (원본 사진 좌표)
* `POINTS`: 클릭 좌표, `LABELS`: 1 포함 / 0 제외

In [ ]:
BOX = None          # 예: [400, 900, 2600, 3300]
POINTS, LABELS = None, None

if BOX:
    m = refine(img, BOX, POINTS, LABELS)
    mask = m if mask is None else Image.fromarray(np.maximum(np.asarray(mask), np.asarray(m)))
assert mask is not None, '마스크가 없어요. 1-6 또는 1-7을 확인하세요.'

cut = cutout(img, mask)
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(img); ax[0].set_title('image')
ax[1].imshow(mask, cmap='gray'); ax[1].set_title('mask (white = food)')
ax[2].imshow(cut); ax[2].set_title('cutout')
for a in ax:
    a.axis('off')
plt.tight_layout(); plt.show()

## 1-8. 다음 단계(VLM)에 넘기기
* `OUT_DIR`에 파일로 저장하고, 같은 내용을 `nukki_out` 변수로도 넘김
* **형식 초안** (주말까지 API 코드로 바꿀 예정)

| 키 | 내용 |
|---|---|
| `image` | 원본 사진 경로 (RGB, EXIF 회전 반영) |
| `mask` | 음식 마스크 경로 (L, 흰색 = 음식, 원본과 같은 크기) |
| `cutout` | 투명 배경 누끼 경로 (RGBA, 원본과 같은 크기) |
| `cutout_crop` | 음식 부분만 잘라낸 누끼 경로 |
| `food_box` | 음식 영역 박스 `[x0, y0, x1, y1]` |
| `dishes` | 선택된 그릇 목록 (id, 이름, 점수, 박스) |

In [ ]:
def to_next_stage(img, mask, cands, select, out_dir=OUT_DIR, name='input'):
    """누끼 결과 → 다음 단계가 읽을 형식 (파일 + dict)"""
    paths = {k: f'{out_dir}/{name}_{k}.png' for k in ('image', 'mask', 'cutout', 'cutout_crop')}
    img.save(paths['image'])
    mask.save(paths['mask'])
    cut = cutout(img, mask)
    cut.save(paths['cutout'])
    cutout(img, mask, crop=True).save(paths['cutout_crop'])
    box = cut.getbbox()
    chosen = [c for c in cands if ((c['id'] in select) if select is not None else c['dish'])]
    out = dict(paths, size=list(img.size), food_box=list(box) if box else None,
               dishes=[{k: c[k] for k in ('id', 'label', 'score', 'box')} for c in chosen],
               finetuned=_m['finetuned'])
    json.dump(out, open(f'{out_dir}/{name}_nukki.json', 'w'), ensure_ascii=False, indent=1)
    return out


nukki_out = to_next_stage(img, mask, cands, SELECT)
print(json.dumps({k: v for k, v in nukki_out.items() if k != 'dishes'}, ensure_ascii=False, indent=1))
print('그릇', len(nukki_out['dishes']), '개 선택')

# 2. VLM (요청 해석)
* 입력: `nukki_out` (1-8 형식) + 사용자 요청 문장
* 출력: sd_prompt, negative_prompt, ad_copy, strategy_reason, aspect_ratio → vlm_out

## 2-1. VLM 모델 & 프로세서 로드 (한 번만 로딩)

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from peft import PeftModel

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
LORA_PATH = "/content/drive/qwen_vl_lora"   # 파인튜닝 어댑터 경로 (개인별 수정)

print("VLM 베이스 모델 로딩 중...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# LoRA 어댑터 있으면 적용 (없으면 베이스만 사용)
if os.path.exists(LORA_PATH):
    print("LoRA 어댑터 적용 중...")
    model = PeftModel.from_pretrained(model, LORA_PATH)
    model = model.merge_and_unload()
    print(f"✅ 파인튜닝 모델 사용: {LORA_PATH}")
else:
    print(f"⚠️ LoRA 어댑터 없음 ({LORA_PATH}) → 베이스 모델로 동작")

model.eval()
processor = AutoProcessor.from_pretrained(model_id)
print("VLM 모델 로딩 완료")

## 2-2. 누끼 단계 연동 (1-8 노트북 결과 활용)
- 세그멘테이션 코드 실행을 통해 얻은 nukki_out과 cutout(투명 배경 또는 회색 처리된 이미지) 경로를 가져옴.
- 만약 파일로 저장되어 있다면 nukki_out['cutout'] 또는 전처리된 이미지를 사용함.


In [ ]:
processed_image_path = nukki_out.get('cutout_crop') or nukki_out.get('cutout', f'{OUT_DIR}/input_cutout_crop.png')

try:
    img_vlm = Image.open(processed_image_path)
    if img_vlm.mode in ('RGBA', 'LA') or (img_vlm.mode == 'P' and 'transparency' in img_vlm.info):
        img_vlm = img_vlm.convert("RGBA")
        gray_bg = Image.new("RGB", img_vlm.size, (128, 128, 128))
        gray_bg.paste(img_vlm, mask=img_vlm.split()[3])

        processed_image_path = f"{OUT_DIR}/vlm_input_processed.jpg"
        gray_bg.save(processed_image_path, format="JPEG")
except Exception as e:
    print(f"이미지 전처리 중 예외 발생 (원본 유지): {e}")

## 2-3. 시스템 프롬프트 및 사용자 입력 세팅

In [ ]:
system_prompt = """You are an expert marketing strategist specializing in helping local restaurant owners boost sales.
Analyze the provided food image (preprocessed with a neutral gray background) and the user's situational context to create an effective promotional strategy.
Generate a detailed English prompt for an AI image generator and a short, impactful Korean ad copy.

Input fields guide:
- If '가게명' (store_name) is '미입력', NEVER invent or include a store name in 'ad_copy'.
- If '대상 고객' (target_audience) is '미입력', infer the best target audience from the food image and request.
- If '원하는 분위기' (mood) is 'AI 추천', choose the most effective mood and explain it in 'strategy_reason'; otherwise, strictly reflect the requested mood in 'sd_prompt'.
- 'aspect_ratio' in your JSON output MUST match the requested '광고 비율'.

You MUST respond ONLY with a valid JSON object matching the exact structure below, without any introductory or concluding text:

{
  "sd_prompt": "Detailed English background description. MUST include 'realistic soft drop shadow under the food, natural directional lighting'",
  "negative_prompt": "text, words, watermark, typography, floating object, bad proportions, unnatural shadows",
  "ad_copy": "Short, catchy marketing copy written in fluent Korean",
  "strategy_reason": "Strategic explanation written in Korean",
  "aspect_ratio": "Choose one from: '1:1', '4:5', or '9:16'"
}"""

def format_user_prompt(form_data: dict) -> str:
    return "\n".join([
        f"상품명: {form_data.get('product_name', '').strip()}",
        f"가게명: {(form_data.get('store_name') or '').strip() or '미입력'}",
        f"광고 요구사항: {form_data.get('request_text', '').strip()}",
        f"대상 고객: {(form_data.get('target_audience') or '').strip() or '미입력'}",
        f"원하는 분위기: {(form_data.get('mood') or '').strip() or 'AI 추천'}",
        f"광고 비율: {(form_data.get('aspect_ratio') or '').strip() or '1:1'}",
    ])

# 프론트엔드 입력값 예시
form_data = {
    "product_name": "샐러드",
    "store_name": "그린모먼트",
    "request_text": "비 와서 손님 없어. 남은 빵이랑 같이 홍보하고 싶어",
    "target_audience": "건강한 샐러드를 즐기는 20-30대 직장인",
    "mood": "AI 추천",
    "aspect_ratio": "4:5"  # '1:1', '4:5', '9:16' 중 택 1
}

user_text = format_user_prompt(form_data)
print("--- VLM 입력 텍스트 ---")
print(user_text)

## 2-4. VLM 프롬프트 포맷팅 및 추론 실행

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": [
        {"type": "image", "image": processed_image_path, "max_pixels": 262144},
        {"type": "text", "text": user_text}
    ]}
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to(model.device)

print("\nVLM 추론 중...")
generated_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]

output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

## 2-5. 결과 파싱 및 vlm_out 생성

In [ ]:
print("\n--- JSON 파싱 결과 ---")

DEFAULT_FALLBACK = {
    "sd_prompt": "a warm cozy restaurant table, realistic soft drop shadow under the food, natural directional lighting, photorealistic",
    "negative_prompt": "text, words, watermark, typography, floating object, bad proportions, unnatural shadows",
    "ad_copy": "오늘의 한 끼",
    "strategy_reason": "fallback",
    "aspect_ratio": form_data.get("aspect_ratio", "1:1"),
}

vlm_out = None
try:
    clean_json_str = output_text.strip().replace("```json", "").replace("```", "").strip()
    vlm_out = json.loads(clean_json_str)
except json.JSONDecodeError:
    print("JSON 파싱 실패. 원본 텍스트:\n", output_text)

# fallback 적용
if not vlm_out or not {"sd_prompt", "ad_copy", "aspect_ratio"} <= vlm_out.keys():
    print("⚠️ fallback 사용")
    vlm_out = DEFAULT_FALLBACK

# 프론트가 명시한 비율 강제
if form_data.get("aspect_ratio") in ("1:1", "4:5", "9:16"):
    vlm_out["aspect_ratio"] = form_data["aspect_ratio"]

# 저장 + 출력
with open(f"{OUT_DIR}/input_vlm.json", "w", encoding="utf-8") as f:
    json.dump(vlm_out, f, ensure_ascii=False, indent=2)
for key, value in vlm_out.items():
    print(f"[{key}]: {value}")

## 핵심 연동 포인트 요약

1. **데이터 전달고리 (`nukki_out`)**:
   * 누끼 노트북의 `1-8` 단계에서 생성되는 `nukki_out['cutout']`(또는 `cutout_crop`) 경로를 VLM 입력 이미지 경로(`processed_image_path`)로 그대로 이어받음.
2. **배경 통일**:
   * 누끼가 완료된 투명 이미지(RGBA)에 회색 배경 처리를 거친 뒤 VLM에 밀어 넣으면, 원본 사진의 불필요한 배경 노이즈가 제거되어 VLM이 음식 형태와 요구사항을 훨씬 정확하게 분석할 수 있음.
3. **다음 단계(`vlm_out`)로 확장**:
   * VLM이 반환한 JSON 결과(`vlm_out`)를 다음 단계인 **3단계(배경 생성 팀의 인페인팅/아웃페인팅)**로 그대로 넘겨주어 전체 파이프라인을 완성할 수 있음.

# 3. 배경 생성 (인페인팅 / 아웃페인팅) — 생성팀
* 입력: `nukki_out['image']`, `nukki_out['mask']`(흰색 = 음식), `vlm_out`
* 출력: 목표 비율의 배경 생성 이미지 → `gen_out`

## 3-1. 설정
* 배경은 **OpenAI 이미지 편집 API**로 생성 (GPU 필요 없음, **호출 1번마다 API 요금 발생**)
* API 키: 드라이브 `AdCanvas_nukki/secrets/openai_api_key.txt` (없으면 환경변수 `OPENAI_API_KEY`). 키를 노트북 코드에 직접 적지 마세요
* `GEN_LONG_SIDE`: 생성 캔버스 긴 변 / `SUBJECT_SCALE`: 캔버스 대비 음식 크기 / `PRESERVE_NATIVE_RES`: 음식을 원본 해상도로 붙일지

In [ ]:
!pip -q install --upgrade openai

import re, base64
from pathlib import Path
from PIL import ImageFilter
from IPython.display import display

GEN_MODEL = "gpt-image-2.5-sunburst"
GEN_LONG_SIDE = 1024          # 생성 캔버스 긴 변 (16의 배수)
SUBJECT_SCALE = 0.62          # 캔버스 대비 음식 크기 비율
PRESERVE_NATIVE_RES = True    # True: 배경을 업스케일하고 음식은 원본 해상도로 합성
MAX_OUTPUT_SIDE = 4096        # 최종 출력 긴 변 상한

# API 키 로드 (없으면 즉시 중단)
API_KEY_PATH = Path(NUKKI_BASE) / 'secrets' / 'openai_api_key.txt'
if API_KEY_PATH.exists():
    os.environ['OPENAI_API_KEY'] = API_KEY_PATH.read_text().strip()

## 3-2. 배경 생성 · 재합성 함수
* 생성 결과 위에 원본 음식 픽셀을 다시 덮어 음식 보존
* 출력: 최종 광고 이미지

**OpenAI 마스크 규칙**: 마스크 PNG에서 알파=0(투명)인 곳이 생성 영역, 음식은 알파=255로 보호

In [ ]:
# ---------- 마스크 연산 ----------
def binarize(m: Image.Image, threshold: int = 127) -> Image.Image:
    a = np.asarray(m.convert("L"), np.uint8)
    return Image.fromarray(np.where(a > threshold, 255, 0).astype(np.uint8), "L")

def _morph(m: Image.Image, radius: int, flt) -> Image.Image:
    # 큰 커널 대신 최대 3px씩 나눠 적용 (정확히 radius만큼)
    out, done = m.convert("L"), 0
    while done < radius:
        step = min(3, radius - done)
        out = out.filter(flt(2 * step + 1))
        done += step
    return out

def dilate(m, radius): return _morph(m, radius, ImageFilter.MaxFilter)
def erode(m, radius):  return _morph(m, radius, ImageFilter.MinFilter)

def feather(m: Image.Image, radius: int) -> Image.Image:
    return m.convert("L") if radius <= 0 else m.convert("L").filter(ImageFilter.GaussianBlur(radius))

def generation_mask_inline(food_mask, protect_px=3, blend_px=6, floor_ratio=0.20, feather_px=4):
    # 반환값: 255 = 배경 생성 영역, 0 = 음식 보호 영역
    b = binarize(food_mask)
    a = np.asarray(b, np.uint8) > 127
    if not a.any():
        raise ValueError("음식 마스크가 비어 있습니다 (알파 채널 확인 필요).")
    protected = np.asarray(dilate(b, protect_px), np.float32)
    shrunk = np.asarray(erode(b, blend_px), np.float32)

    # 바닥면(아래쪽 floor_ratio 구간)은 보호 영역을 줄여 접지 그림자가 자연스럽게 생성되도록 함
    ys, _ = np.where(a)
    top, bottom = ys.min(), ys.max()
    split = bottom - (bottom - top) * float(np.clip(floor_ratio, 0, 1))
    band = max(8.0, feather_px * 3.0)
    rows = np.arange(b.height, dtype=np.float32)
    w = np.clip((rows - (split - band / 2)) / band, 0, 1)[:, None]

    blended = np.clip(protected * (1 - w) + shrunk * w, 0, 255).astype(np.uint8)
    return feather(Image.fromarray(255 - blended, "L"), feather_px)

def to_openai_mask(image: Image.Image, gen_mask: Image.Image) -> Image.Image:
    # OpenAI 규칙: 알파=0 → 편집(생성), 알파=255 → 보존
    m = image.convert("RGBA")
    m.putalpha(ImageOps.invert(binarize(gen_mask)))
    return m

# ---------- 비율 / 캔버스 ----------
def parse_ratio(v):
    if isinstance(v, (list, tuple)) and len(v) == 2:
        rw, rh = float(v[0]), float(v[1])
    elif isinstance(v, str):
        parts = [p for p in re.split(r"\s*[:x/×,]\s*", v.strip()) if p]
        if len(parts) != 2:
            raise ValueError(f"비율 형식을 해석할 수 없습니다: {v!r}")
        rw, rh = float(parts[0]), float(parts[1])
    elif isinstance(v, (int, float)):
        rw, rh = float(v), 1.0
    else:
        raise ValueError(f"비율 형식을 해석할 수 없습니다: {v!r}")
    if rw <= 0 or rh <= 0:
        raise ValueError(f"비율은 양수여야 합니다: {v!r}")
    return rw, rh

def _snap16(v): return max(16, int(round(v / 16)) * 16)

def canvas_size_for_ratio(ratio, long_side=1024):
    rw, rh = ratio
    if rw >= rh:
        return _snap16(long_side), _snap16(long_side * rh / rw)
    return _snap16(long_side * rw / rh), _snap16(long_side)

def build_canvas(src_rgba: Image.Image, cw: int, ch: int, subject_scale: float) -> dict:
    alpha = src_rgba.getchannel("A")
    bbox = alpha.point(lambda v: 255 if v > 0 else 0).getbbox()
    if bbox is None:
        raise ValueError("알파 채널이 비어 있습니다.")
    src = src_rgba.crop(bbox)            # 투명 여백 제거
    sw, sh = src.size
    scale = subject_scale * min(cw / sw, ch / sh)
    pw, ph = max(1, round(sw * scale)), max(1, round(sh * scale))
    cx, cy = (cw - pw) // 2, (ch - ph) // 2

    fg = src.resize((pw, ph), Image.LANCZOS)   # RGBA 리사이즈는 premultiplied로 처리되어 테두리 번짐 없음
    canvas = Image.new("RGB", (cw, ch), (127, 127, 127))
    canvas.paste(fg.convert("RGB"), (cx, cy), fg)   # 알파를 마스크로 사용

    food_mask = Image.new("L", (cw, ch), 0)
    food_mask.paste(binarize(fg.getchannel("A")), (cx, cy))
    return {"src": src, "fg": fg, "canvas": canvas, "food_mask": food_mask,
            "box": (cx, cy, cx + pw, cy + ph), "scale": scale}

# ---------- OpenAI 호출 ----------
def edit_via_openai(image: Image.Image, gen_mask: Image.Image, prompt_text: str, model: str = None) -> Image.Image:
    from openai import OpenAI
    client = OpenAI()

    img_buf = io.BytesIO(); image.convert("RGBA").save(img_buf, format="PNG"); img_buf.seek(0)
    mask_buf = io.BytesIO(); to_openai_mask(image, gen_mask).save(mask_buf, format="PNG"); mask_buf.seek(0)
    img_buf.name, mask_buf.name = "image.png", "mask.png"

    response = client.images.edit(
        model=model or GEN_MODEL,
        prompt=prompt_text,
        image=img_buf,
        mask=mask_buf,
        size=f"{image.width}x{image.height}",
        quality="high",
    )
    out = Image.open(io.BytesIO(base64.b64decode(response.data[0].b64_json))).convert("RGB")
    if out.size != image.size:
        print(f"  안내: 반환 크기 {out.size} → 캔버스 크기 {image.size}로 맞춤")
        out = out.resize(image.size, Image.LANCZOS)
    return out

# ---------- 재합성 (4단계에서 사용) ----------
def paste_back(generated: Image.Image, layout: dict, preserve_native=True, max_side=4096):
    cw, ch = generated.size
    up = 1.0
    if preserve_native:
        up = max(1.0, 1.0 / layout["scale"])
        if max(cw, ch) * up > max_side:
            up = max(1.0, max_side / max(cw, ch))
    W, H = round(cw * up), round(ch * up)
    bg = generated.resize((W, H), Image.LANCZOS) if (W, H) != (cw, ch) else generated.copy()

    cx, cy = layout["box"][:2]
    pos = (round(cx * up), round(cy * up))
    src, fg_small = layout["src"], layout["fg"]
    fw, fh = round(fg_small.width * up), round(fg_small.height * up)
    if abs(fw - src.width) <= 1 and abs(fh - src.height) <= 1:
        fg = src                                   # 원본 픽셀 그대로
    elif (fw, fh) == fg_small.size:
        fg = fg_small
    else:
        fg = src.resize((fw, fh), Image.LANCZOS)
    bg.paste(fg.convert("RGB"), pos, fg.getchannel("A"))
    return bg, fg, pos

# ---------- 평가 (4단계에서 사용) ----------
def evaluate(final: Image.Image, fg: Image.Image, pos, src: Image.Image) -> dict:
    x, y = pos
    region = np.asarray(final.crop((x, y, x + fg.width, y + fg.height)).convert("RGB"), np.float32)
    ref = np.asarray(fg.convert("RGB"), np.float32)
    core = np.asarray(fg.getchannel("A")) == 255   # 완전 불투명 음식 픽셀
    diff = np.abs(region - ref)[core]
    mae = float(diff.mean()) if diff.size else float("nan")
    return {
        "food_mae": round(mae, 4),
        "food_max_abs_err": float(diff.max()) if diff.size else float("nan"),
        "food_pixels_checked": int(core.sum()),
        "food_native_resolution": fg.size == src.size,
        "food_size": list(fg.size),
        "output_size": list(final.size),
        "food_preserved": bool(diff.size and mae < 0.5),
    }

def review_sheet(canvas, gen_mask, generated, final, height=512):
    tiles = [canvas, Image.merge("RGB", [gen_mask] * 3), generated, final.convert("RGB")]
    tiles = [t.resize((round(t.width * height / t.height), height), Image.LANCZOS) for t in tiles]
    sheet = Image.new("RGB", (sum(t.width for t in tiles) + 8 * (len(tiles) - 1), height), (255, 255, 255))
    x = 0
    for t in tiles:
        sheet.paste(t, (x, 0)); x += t.width + 8
    return sheet

print("배경 생성 함수 등록 완료")

## 3-3. 입력 준비 (`nukki_out` + `vlm_out`) → 캔버스 · 생성 마스크 미리보기
* 음식: `nukki_out['image']` + `nukki_out['mask']`(흰색 = 음식)로 RGBA 누끼를 만들어 사용
* 비율: `vlm_out['aspect_ratio']` (`'1:1'`, `'4:5'`, `'9:16'`)
* 프롬프트: `vlm_out['sd_prompt']` + 음식 보존 지시 + `negative_prompt`(OpenAI 편집 API에는 negative 입력이 없어서 `Avoid: ...` 문장으로 붙임)
* 미리보기: **캔버스 | 생성 마스크(흰색 = 생성) | OpenAI 전송 마스크(체크무늬 = 투명 = 생성)**

In [ ]:
PRESERVE_TEXT = ("The food and its dish in the center are the hero subject and must remain exactly unchanged; "
                 "only generate the surrounding background and tabletop.")


def build_gen_prompt(vlm: dict) -> str:
    """vlm_out → OpenAI 편집 프롬프트 (sd_prompt + 음식 보존 지시 + negative를 Avoid 문장으로)"""
    parts = [vlm.get('sd_prompt', '').strip().rstrip('.') + '.', PRESERVE_TEXT]
    neg = (vlm.get('negative_prompt') or '').strip().rstrip('.')
    if neg:
        parts.append(f"Avoid: {neg}.")
    return ' '.join(parts)


def load_food_rgba(nukki: dict) -> Image.Image:
    """nukki_out의 원본 + 마스크 → RGBA 누끼 (흰색 마스크 = 불투명)"""
    rgba = Image.open(nukki['image']).convert('RGB')
    rgba.putalpha(Image.open(nukki['mask']).convert('L').resize(rgba.size, Image.NEAREST))
    return rgba


try:
    GEN_RATIO = parse_ratio(vlm_out.get('aspect_ratio') or '1:1')
except ValueError as e:
    print(f"비율 오류: {e} → 1:1 사용")
    GEN_RATIO = (1.0, 1.0)
GEN_PROMPT = build_gen_prompt(vlm_out)
CW, CH = canvas_size_for_ratio(GEN_RATIO, GEN_LONG_SIDE)

food_rgba = load_food_rgba(nukki_out)
layout = build_canvas(food_rgba, CW, CH, SUBJECT_SCALE)
gen_mask = generation_mask_inline(layout['food_mask'], protect_px=3, blend_px=6, feather_px=4)

print(f"비율: {vlm_out.get('aspect_ratio')} → 캔버스 {CW}x{CH} | 음식 축소 비율 {layout['scale']:.3f}")
print(f"프롬프트: {GEN_PROMPT}")

api_mask = to_openai_mask(layout['canvas'], gen_mask)
checker = Image.fromarray(((np.indices((CH, CW)).sum(0) // 16 % 2) * 60 + 160).astype(np.uint8)).convert('RGB')
checker.paste(api_mask.convert('RGB'), (0, 0), api_mask.getchannel('A'))
display(Image.fromarray(np.hstack([np.array(layout['canvas']),
                                   np.array(Image.merge('RGB', [gen_mask] * 3)),
                                   np.array(checker)])))

## 3-4. 배경 생성 (OpenAI API 호출) → `gen_out`
* **이 셀을 실행할 때마다 API 요금 발생**. 미리보기(3-3)를 확인한 뒤 실행
* 결과는 `OUT_DIR`에 저장하고, 같은 내용을 `gen_out` 변수로 4단계에 넘김


In [ ]:
t0 = time.time()
print("OpenAI 배경 생성 요청 중...")
generated_raw = edit_via_openai(layout['canvas'], gen_mask, GEN_PROMPT)
print(f"생성 완료 | {time.time() - t0:.1f}초")

gen_paths = {k: f'{OUT_DIR}/input_{k}.png' for k in ('canvas', 'gen_mask', 'generated')}
layout['canvas'].save(gen_paths['canvas'])
gen_mask.save(gen_paths['gen_mask'])
generated_raw.save(gen_paths['generated'])

gen_out = dict(gen_paths, ratio=vlm_out.get('aspect_ratio'), canvas_size=[CW, CH],
               food_box=list(layout['box']), scale=layout['scale'], prompt=GEN_PROMPT, model=GEN_MODEL)
json.dump(gen_out, open(f'{OUT_DIR}/input_gen.json', 'w'), ensure_ascii=False, indent=1)

fig, ax = plt.subplots(1, 2, figsize=(12, 7))
ax[0].imshow(layout['canvas']); ax[0].set_title('canvas')
ax[1].imshow(generated_raw); ax[1].set_title('generated (model output)')
for a in ax:
    a.axis('off')
plt.tight_layout(); plt.show()

## 4. 원본 음식 재합성 — 생성팀
* 생성 결과 위에 원본 음식 픽셀을 다시 덮어 음식 보존
* 출력: 최종 광고 이미지

## 4-1. 원본 음식 재합성 + 평가
* 생성된 배경 위에 누끼를 **알파 채널 그대로** 다시 덮어서, 모델이 음식을 조금 바꿨더라도 원본 픽셀로 되돌림
* `PRESERVE_NATIVE_RES=True`: 배경을 업스케일하고 음식은 **원본 해상도 픽셀 그대로** 붙임 (긴 변 `MAX_OUTPUT_SIDE` 이하)
* 평가: `food_mae` = 완전 불투명 음식 픽셀에서 최종 결과와 원본의 평균 차이 (0이면 완전 보존, 0.5 미만이면 통과)

In [ ]:
final_img, fg_used, fg_pos = paste_back(generated_raw, layout,
                                        preserve_native=PRESERVE_NATIVE_RES, max_side=MAX_OUTPUT_SIDE)
metrics = evaluate(final_img, fg_used, fg_pos, layout['src'])
metrics.update(model=GEN_MODEL, ratio=vlm_out.get('aspect_ratio'), canvas_size=[CW, CH], prompt=GEN_PROMPT)

status = '통과' if metrics['food_preserved'] else '확인 필요'
print(f"food_mae={metrics['food_mae']} / 원본 해상도={metrics['food_native_resolution']} "
      f"/ 출력 {metrics['output_size'][0]}x{metrics['output_size'][1]} → {status}")
review = review_sheet(layout['canvas'], gen_mask, generated_raw, final_img)
display(review)   # 캔버스 | 생성 마스크 | 모델 원본 출력 | 최종 결과

## 4-2. 최종 결과 저장 → `final_out` (5. 서버로 넘기기)

In [ ]:
final_paths = {'final': f'{OUT_DIR}/input_final.png', 'review': f'{OUT_DIR}/input_review.png'}
final_img.save(final_paths['final'])
review.save(final_paths['review'])
json.dump(metrics, open(f'{OUT_DIR}/input_metrics.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)

final_out = dict(final_paths, metrics=metrics, ad_copy=vlm_out.get('ad_copy', ''),
                 aspect_ratio=vlm_out.get('aspect_ratio'))
json.dump(final_out, open(f'{OUT_DIR}/input_final.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

display(Image.open(final_paths['final'])) # 최종 결과 (input_final.png)
print('저장:', final_paths['final'])
print('광고 문구:', final_out['ad_copy'])

# 5. 서버 (FastAPI + ngrok) — 주말 작성 예정
* 프론트 `POST /generate` → 1~4 실행 → `{imageUrl, copy}` 응답